# Class 9. Module & Package System

`import`가 모듈을 찾고 실행하는 과정을 확인한다. 코드를 Package로 묶고, `__name__`, `__init__.py`, 절대·상대 Import를 사용해 본다. pip와 가상환경으로 프로젝트의 의존성도 관리한다.


## 1. Module

### 1.1 Module이란?

Module은 관련된 코드와 이름을 묶는 단위다. 직접 작성할 때는 보통 `.py` 파일 하나로 만든다. 코드를 여러 곳에서 재사용하고, 모듈별 Namespace로 이름을 나누는 데 쓰인다.

Python 내장 모듈, 표준 라이브러리, 외부 패키지로 설치한 모듈, 직접 작성한 모듈을 아래 예제에서 비교한다.


In [ ]:
# Module 예시
print("=== Module 종류 ===")

# 1. Built-in Module
import sys
print(f"1. Built-in: sys.version = {sys.version[:20]}...")

# 2. Standard Library
import json
data = json.dumps({'name': 'Alice', 'age': 25})
print(f"2. Standard Library: json.dumps() = {data}")

# 3. Third-party (설치되어 있다면)
try:
    import numpy as np
    print(f"3. Third-party: numpy version = {np.__version__}")
except ImportError:
    print("3. Third-party: numpy가 설치되지 않음")

# 4. User-defined는 파일로 작성 필요

### 1.2 import 메커니즘

#### import가 실행될 때 일어나는 일

```python
import my_module
```

**1단계**: Module 찾기 (Search)
- `sys.path`에 있는 경로들을 순서대로 검색
- `my_module.py` 파일 찾기

**2단계**: Module 컴파일 (Compile)
- `.py` → `.pyc` (bytecode)
- `__pycache__` 디렉토리에 저장

**3단계**: Module 실행 (Execute)
- 모듈의 모든 코드 실행
- Module Namespace 생성

**4단계**: Module Caching (Cache)
- `sys.modules`에 저장
- 다음 import부터는 캐시 사용

**5단계**: Name Binding (Bind)
- 현재 Namespace에 모듈 이름 바인딩


In [ ]:
# import 메커니즘 확인
import sys

print("=== import 메커니즘 ===")

# 1. Module Search Path
print("\n1. sys.path (Module 검색 경로):")
for i, path in enumerate(sys.path[:3], 1):
    print(f"   {i}. {path}")
print("   ...")

# 2. Module Cache
print("\n2. sys.modules (캐시된 Module):")
cached_modules = list(sys.modules.keys())[:5]
for module in cached_modules:
    print(f"   - {module}")
print(f"   ... (총 {len(sys.modules)}개)")

# 3. import 후 확인
import math
print(f"\n3. math import 후:")
print(f"   'math' in sys.modules: {'math' in sys.modules}")
print(f"   'math' in dir(): {'math' in dir()}")

### 1.3 import 방식

#### 다양한 import 문법


In [ ]:
# import 방식
print("=== import 방식 ===")

# 1. 기본 import
import math
print(f"1. import math")
print(f"   math.pi = {math.pi}")

# 2. as로 별칭
import numpy as np  # numpy가 없으면 주석 처리
# print(f"\n2. import numpy as np")
# print(f"   np.array([1, 2, 3]) = {np.array([1, 2, 3])}")

# 3. from import (특정 항목만)
from math import sqrt, pi
print(f"\n3. from math import sqrt, pi")
print(f"   sqrt(16) = {sqrt(16)}")
print(f"   pi = {pi}")

# 4. from import as (별칭)
from math import factorial as fact
print(f"\n4. from math import factorial as fact")
print(f"   fact(5) = {fact(5)}")

# 5. from import * (비권장!)
# from math import *
print("\n5. from math import * (비권장)")
print("   → Namespace 오염 위험")

#### import 방식 비교

| 방식 | 이름을 사용하는 방법 | 확인할 점 |
| --- | --- | --- |
| `import module` | `module.name` | 모듈 이름으로 출처를 알 수 있다. |
| `import module as alias` | `alias.name` | 별칭은 프로젝트의 관례에 맞춘다. |
| `from module import name` | `name` | 현재 Namespace의 이름과 겹치는지 확인한다. |
| `from module import *` | 공개된 이름을 한꺼번에 가져온다. | 출처와 이름 충돌을 파악하기 어려워 피하는 편이 좋다. |


### 1.4 `__name__` 변수

파일을 Script로 직접 실행하면 `__name__`은 `"__main__"`이다. Import해서 사용하면 모듈 이름이 된다. 이를 이용해 직접 실행할 때만 동작할 코드를 구분한다.


In [ ]:
# __name__ 확인
print("=== __name__ 변수 ===")
print(f"현재 __name__: {__name__}")

# Jupyter Notebook에서는 '__main__'이 아닐 수 있음
if __name__ == "__main__":
    print("이 코드는 직접 실행되었습니다")
else:
    print(f"이 코드는 '{__name__}' 모듈로 import되었습니다")

#### `__name__`의 활용

**패턴**: Module로도 쓰고 Script로도 실행

```python
# calculator.py

def add(a, b):
    return a + b

def subtract(a, b):
    return a - b

# Module로 import될 때는 실행 안 됨

if __name__ == "__main__":
    # Script로 실행될 때만 실행
    print("Calculator Test")
    print(f"5 + 3 = {add(5, 3)}")
    print(f"5 - 3 = {subtract(5, 3)}")
```

**사용 예**:
```bash
# Script로 실행

$ python calculator.py
Calculator Test
5 + 3 = 8
5 - 3 = 2

# Module로 import

>>> import calculator
>>> calculator.add(10, 20)
30
```


### 1.5 Module Search Path

#### `sys.path`의 구성

**Module 검색 순서**:
1. 현재 디렉토리
2. `PYTHONPATH` 환경 변수
3. 표준 라이브러리 경로
4. site-packages (pip 설치 경로)


In [ ]:
# sys.path 확인
import sys

print("=== Module Search Path ===")
print("\nsys.path:")
for i, path in enumerate(sys.path, 1):
    print(f"{i}. {path}")

# 동적으로 경로 추가
print("\n경로 추가:")
new_path = "/my/custom/path"
sys.path.append(new_path)
print(f"sys.path에 '{new_path}' 추가됨")
print(f"마지막 경로: {sys.path[-1]}")

### 1.6 Module Caching

Import한 모듈은 `sys.modules`에 저장된다. 같은 모듈을 다시 Import하면 보통 저장된 모듈 객체를 사용하므로 최상위 코드가 다시 실행되지 않는다.


In [ ]:
# Module Caching 확인
import sys

print("=== Module Caching ===")

# math를 여러 번 import
import math
print(f"첫 import: id(math) = {id(math)}")

import math
print(f"두 번째 import: id(math) = {id(math)}")
print("→ 같은 객체 (캐시 사용)")

# sys.modules 확인
print(f"\n'math' in sys.modules: {'math' in sys.modules}")
print(f"sys.modules['math']: {sys.modules['math']}")

# Module 재로딩 (개발 시에만)
import importlib
print("\nModule 재로딩:")
importlib.reload(math)
print("math 모듈 재로딩됨")

## 2. Package 구조화

### 2.1 Package vs Module

#### 개념 정리

**Module**: 단일 `.py` 파일
```
calculator.py
```

**Package**: Module들을 담은 디렉토리
```
my_package/
    __init__.py
    module1.py
    module2.py
```

**Package의 특징**:
- 디렉토리에 `__init__.py` 포함 (Python 3.3+는 선택)
- 계층적 구조 가능
- Namespace로 구조화


### 2.2 Package 구조 예시

#### 전형적인 Package 구조

```
my_project/
│
├── my_package/              # Package 루트
│   ├── __init__.py         # Package 초기화
│   ├── module1.py          # Module 1
│   ├── module2.py          # Module 2
│   │
│   └── sub_package/        # Sub-package
│       ├── __init__.py
│       └── module3.py
│
├── tests/                  # 테스트
│   └── test_module1.py
│
├── setup.py                # 설치 스크립트
├── requirements.txt        # 의존성
└── README.md               # 문서
```


### 2.3 `__init__.py`의 역할

#### `__init__.py`가 하는 일

**1. Package 표시** (Python 3.3 이전 필수)
```python
# 빈 파일이어도 OK

# my_package/__init__.py

```

**2. Package 초기화 코드**
```python
# my_package/__init__.py

print("Package 초기화")
VERSION = "1.0.0"
```

**3. 편의 Import**
```python
# my_package/__init__.py

from .module1 import func1
from .module2 import func2

# 사용자는 간단하게

from my_package import func1  # 대신 from my_package.module1 import func1
```

**4. `__all__` 정의**
```python
# my_package/__init__.py

__all__ = ['func1', 'func2']
```


### 2.4 Absolute vs Relative Import

#### Absolute Import (절대 Import)

Package 이름부터 모듈 경로를 적는다. 이름만으로 출처를 파악하기 쉽지만, Package 이름이 길면 Import 문도 길어진다.

```python
# my_package/module1.py

from my_package.module2 import func2
from my_package.sub_package.module3 import func3
```


#### Relative Import (상대 Import)

현재 Package를 기준으로 모듈을 지정한다. `.`은 현재 Package, `..`은 상위 Package를 뜻한다. Package 내부의 경로를 짧게 적을 수 있지만, 파일을 Script로 직접 실행할 때는 Package 정보가 없어 오류가 날 수 있다.

```python
# my_package/module1.py

from . import module2
from .module2 import func2
from ..other_package import something
from .sub_package import module3
```


#### Import 방식 선택

절대 Import는 이름만으로 모듈의 위치를 읽기 쉽다. 상대 Import는 Package 내부의 관계를 짧게 나타낼 수 있다. 프로젝트의 구조와 기존 스타일에 맞추되, 이름의 출처를 확인하기 어려운 Wildcard Import는 피한다.

```python
from my_package.utils import helper
from .utils import helper
```


### 2.5 `__all__` 변수

#### `__all__`의 역할

정의: `from module import *` 시 import될 이름 목록

```python
# module.py

def public_func():
    pass

def _private_func():
    pass

__all__ = ['public_func']  # 이것만 export
```

효과:
```python
from module import *
# public_func만 import됨

# _private_func는 import 안 됨

```


### 2.6 Namespace Package (Python 3.3+)

Namespace Package는 `__init__.py` 없이 구성할 수 있으며, 서로 다른 경로에 있는 Package 부분을 하나의 Namespace로 묶을 수 있다. 여러 배포 패키지가 같은 Namespace를 사용하는 구조에서 활용한다.

```text
my_namespace/
    part1/
        module1.py
    part2/
        module2.py
```

Namespace Package 자체에는 `__init__.py`의 초기화 코드를 둘 수 없다.


## 3. Package Management

### 3.1 pip 기초

#### pip란?

**pip**: Python Package Installer

**주요 명령어**:
```bash
# Package 설치

pip install package_name

# 특정 버전 설치

pip install package_name==1.0.0

# 업그레이드

pip install --upgrade package_name

# 제거

pip uninstall package_name

# 설치된 Package 목록

pip list

# Package 정보

pip show package_name

# requirements.txt로 설치

pip install -r requirements.txt

# requirements.txt 생성

pip freeze > requirements.txt
```


In [ ]:
# pip 명령어 (Jupyter에서 실행)
print("=== pip 기초 ===")

# 설치된 Package 목록 (일부)
import subprocess

result = subprocess.run(['pip', 'list'], capture_output=True, text=True)
lines = result.stdout.split('\n')[:10]
print("\n설치된 Package (상위 10개):")
for line in lines:
    print(line)
print("...")

### 3.2 requirements.txt

`requirements.txt`에는 설치할 Package와 버전 조건을 적는다.

예시:
```txt
# requirements.txt

numpy==1.21.0
pandas>=1.3.0
matplotlib
requests~=2.26.0
```

**버전 지정 방식**:
- `==1.0.0` : 정확한 버전
- `>=1.0.0` : 이상
- `<=1.0.0` : 이하
- `~=1.0.0` : 호환 버전 (1.0.x)
- 버전 없음: 최신 버전

사용법:
```bash
# 현재 환경의 Package 목록 저장

pip freeze > requirements.txt

# requirements.txt로 설치

pip install -r requirements.txt
```


### 3.3 Virtual Environment

프로젝트마다 필요한 라이브러리 버전이 다를 수 있다. 가상환경을 나누면 프로젝트 A와 B에 서로 다른 버전을 설치하고 사용할 수 있다.

```text
프로젝트 A → venv_A → A에서 사용할 패키지와 버전
프로젝트 B → venv_B → B에서 사용할 패키지와 버전
```


#### venv 사용법

**Python 내장 도구**:

```bash
# 1. 가상환경 생성

python -m venv myenv

# 2. 활성화

# Windows

myenv\Scripts\activate

# macOS/Linux

source myenv/bin/activate

# 3. Package 설치

(myenv) $ pip install numpy pandas

# 4. 비활성화

(myenv) $ deactivate
```

**virtualenv (대안)**:
```bash
pip install virtualenv
virtualenv myenv
```


### 3.4 conda

#### conda vs pip

| 특징 | pip | conda |
|------|-----|-------|
| 관리 대상 | Python Package | 모든 Package (C, R 등) |
| 가상환경 | venv 별도 | 통합 제공 |
| 생태계 | PyPI | Anaconda |

**conda 기본 명령어**:
```bash
# 환경 생성

conda create -n myenv python=3.9

# 환경 활성화

conda activate myenv

# Package 설치

conda install numpy pandas

# 환경 목록

conda env list

# 환경 삭제

conda env remove -n myenv

# environment.yml로 환경 생성

conda env create -f environment.yml

# environment.yml 생성

conda env export > environment.yml
```


#### conda vs mamba

`mamba`도 conda 환경과 패키지를 관리하는 데 사용할 수 있다.

```bash
# mamba 설치

conda install mamba -c conda-forge

# conda 대신 mamba 사용 (명령어 동일)

mamba install numpy
mamba create -n myenv python=3.9
```

위 예제의 `install`, `create`는 conda와 같은 형태로 사용한다.


### 3.5 프로젝트 환경 관리

프로젝트마다 가상환경을 나눈다.
```bash
# 프로젝트마다 독립적인 환경

python -m venv venv
source venv/bin/activate
```

설치한 Package 목록을 `requirements.txt`로 관리한다.
```bash
# 개발 시작

pip install -r requirements.txt

# 새 Package 설치 후

pip freeze > requirements.txt
```

동일한 환경을 재현하려면 버전 조건을 확인한다.
```txt
# 프로덕션

numpy==1.21.0  # 정확한 버전

# 개발

numpy>=1.21.0  # 유연한 버전
```

환경 디렉토리와 생성 파일은 `.gitignore`에 추가한다.
```gitignore
venv/
__pycache__/
*.pyc
.env
```

필요하면 개발·운영·테스트용 의존성을 나눈다.
```txt
requirements-dev.txt   # 개발용
requirements-prod.txt  # 프로덕션용
requirements-test.txt  # 테스트용
```


## 종합 실습

### 실습 1: Simple Package 만들기

#### 목표: 기본 Package 구조 이해

**디렉토리 구조**:
```
calculator_package/
    __init__.py
    basic.py
    advanced.py
```

**코드**:
```python
# calculator_package/__init__.py

"""Calculator Package"""
from .basic import add, subtract
from .advanced import power, sqrt

__version__ = "1.0.0"
__all__ = ['add', 'subtract', 'power', 'sqrt']

# calculator_package/basic.py

def add(a, b):
    """Add two numbers"""
    return a + b

def subtract(a, b):
    """Subtract two numbers"""
    return a - b

# calculator_package/advanced.py

def power(base, exp):
    """Calculate power"""
    return base ** exp

def sqrt(x):
    """Calculate square root"""
    return x ** 0.5
```

**사용**:
```python
# 방법 1

from calculator_package import add, power
print(add(2, 3))      # 5
print(power(2, 3))    # 8

# 방법 2

import calculator_package as calc
print(calc.add(2, 3))     # 5
print(calc.__version__)   # 1.0.0
```


### 실습 2: requirements.txt 실습

**시나리오**: 데이터 분석 프로젝트

```txt
# requirements.txt

numpy==1.21.0
pandas==1.3.0
matplotlib==3.4.2
seaborn==0.11.1
jupyter==1.0.0
```

**설치**:
```bash
# 가상환경 생성

python -m venv data_env

# 활성화

source data_env/bin/activate  # macOS/Linux
data_env\Scripts\activate     # Windows

# Package 설치

pip install -r requirements.txt

# 확인

pip list
```


## 복습과 토론

1. `sys.path`와 `sys.modules`는 Import 과정에서 각각 어떤 역할을 하는가?
2. 모듈을 직접 실행할 때와 Import할 때 `__name__` 값을 비교해 보자.
3. `__init__.py`에서 이름을 Import하면 Package 사용 코드는 어떻게 달라지는가?
4. 절대 Import, 상대 Import, Wildcard Import의 이름 해석과 가독성을 비교해 보자.
5. 새 가상환경을 만들고 `requirements.txt`로 패키지를 설치하는 과정을 적어보자.
6. pip와 conda가 관리하는 대상과 환경 구성 방법은 어떻게 다른가?

## 참고 자료

- Module Search Path: https://dsaint31.tistory.com/528
- Package Manager: https://dsaint31.tistory.com/534
- pip 사용법: https://ds31x.tistory.com/16
- conda: https://ds31x.blogspot.com/2023/07/env-conda-anaconda-and-miniconda.html
- PyPI: https://pypi.org/
